# PM2.5 Forecasting – Part 1
**Data audit, cleaning, exploration and preprocessing**

Master 1 Mini Project 2026–2027

## 0. Setup

In [1]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import pacf, adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

SEED = 42
DATA_FILE = "combined_gee_pm25.csv"
OUTPUT_DIR = Path("results")
MIN_OBS_PER_STATION = 100
MIN_RUN_DAYS = 20
TEST_START = "2025-01-01"
MODEL_ID = "eda_pm25"
MODEL_GROUP = "G18"

OUTPUT_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")

DATA_PATH = Path(DATA_FILE) if Path(DATA_FILE).exists() else Path("/content") / DATA_FILE
raw = pd.read_csv(DATA_PATH)
raw["date"] = pd.to_datetime(raw["date"])
print(raw.shape)
raw.head()

ModuleNotFoundError: No module named 'numpy'

## 1. Dataset audit

### 1.1 General information

In [ ]:
audit = pd.Series({
    "source": "Kaggle - Air Quality, Mediterranean Basin (2016-2026)",
    "license": "to be checked on the Kaggle dataset page",
    "rows": len(raw),
    "columns": raw.shape[1],
    "first date": raw["date"].min().date(),
    "last date": raw["date"].max().date(),
    "stations (sensor_name)": raw["sensor_name"].nunique(),
    "stations (GEE id)": raw["id"].nunique(),
    "latitude range": f"{raw['latitude'].min():.2f} to {raw['latitude'].max():.2f}",
    "longitude range": f"{raw['longitude'].min():.2f} to {raw['longitude'].max():.2f}",
}, name="value")
audit.to_frame()

In [ ]:
units = {
    "pm25": "µg/m³",
    "AOD": "dimensionless",
    "NO2": "mol/m² (Sentinel-5P column)",
    "CO": "mol/m² (Sentinel-5P column)",
    "O3": "mol/m² (Sentinel-5P column)",
    "temperature_celsius": "°C",
    "pressure_mb": "mb",
    "wind_u": "m/s",
    "wind_v": "m/s",
    "latitude": "degrees",
    "longitude": "degrees",
}
pd.DataFrame({"unit": units}).rename_axis("variable")

The units of NO2, CO and O3 are not given in the file. They are assumed to be the Sentinel-5P column densities (mol/m²) from their order of magnitude.

### 1.2 Station identifiers, timestamps and duplicates

In [ ]:
ids_per_name = raw.groupby("sensor_name")["id"].nunique()
print("sensor_name values with more than one id:", (ids_per_name > 1).sum())
print("ids with more than one sensor_name:", (raw.groupby("id")["sensor_name"].nunique() > 1).sum())
print("max number of coordinates for one id:", raw.groupby("id")[["latitude", "longitude"]].nunique().max().max())
print("datetime_utc is the same day as date:", (pd.to_datetime(raw["datetime_utc"]).dt.normalize() == raw["date"]).all())
print()
print("Exact duplicate rows:", raw.duplicated().sum())
print("Duplicates on (date, latitude, longitude, id):", raw.duplicated(["date", "latitude", "longitude", "id"]).sum())
print("Duplicates on (date, id):", raw.duplicated(["date", "id"]).sum())
print("Duplicates on (date, sensor_name):", raw.duplicated(["date", "sensor_name"]).sum())
raw[raw["sensor_name"].isin(ids_per_name[ids_per_name > 1].index)].groupby(["sensor_name", "id"])[["latitude", "longitude"]].first()

## 2. Data cleaning

In [ ]:
print("Missing values per column (raw data):")
raw.isna().sum()[lambda s: s > 0].to_frame("missing")

In [ ]:
n_missing_pm25 = int(raw["pm25"].isna().sum())
n_low = int((raw["pm25"] <= 0).sum())
n_high = int((raw["pm25"] >= 500).sum())

valid = raw["pm25"].notna() & (raw["pm25"] > 0) & (raw["pm25"] < 500)
df = raw[valid].drop(columns="datetime_utc").sort_values(["id", "date"]).reset_index(drop=True)
n_removed = len(raw) - len(df)

pd.Series({
    "rows before cleaning": len(raw),
    "missing PM2.5": n_missing_pm25,
    "PM2.5 <= 0": n_low,
    "PM2.5 >= 500": n_high,
    "rows removed": n_removed,
    "rows after cleaning": len(df),
}, name="count").to_frame()

## 3. Exploratory data analysis

In [ ]:
predictors = ["AOD", "NO2", "CO", "O3", "temperature_celsius", "pressure_mb", "wind_u", "wind_v"]
analysis_cols = ["pm25"] + predictors

### 3.1 Descriptive statistics

In [ ]:
summary = df[analysis_cols].describe().T
summary["skewness"] = df[analysis_cols].skew()
summary["kurtosis"] = df[analysis_cols].kurtosis()
summary

In [ ]:
obs_per_station = df.groupby("id").size()
print("Stations (id):", df["id"].nunique(), "| station names:", df["sensor_name"].nunique())
print("Observations per station:")
obs_per_station.describe()[["min", "25%", "50%", "75%", "max"]]

### 3.2 Distributions

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 9))
for ax, col in zip(axes.flat, analysis_cols):
    sns.histplot(df[col], bins=40, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 9))
for ax, col in zip(axes.flat, analysis_cols):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

### 3.3 Target: skewness and high concentrations

In [ ]:
pm = df["pm25"]
print(f"skewness = {pm.skew():.2f}, kurtosis = {pm.kurtosis():.2f}")
print(f"mean = {pm.mean():.2f}, median = {pm.median():.2f}, max = {pm.max():.1f}")
for threshold in [25, 50]:
    print(f"PM2.5 > {threshold}: {(pm > threshold).sum()} observations ({(pm > threshold).mean() * 100:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(pm, bins=60, ax=axes[0])
axes[0].set_title("PM2.5")
sns.histplot(np.log1p(pm), bins=60, ax=axes[1])
axes[1].set_title("log(1 + PM2.5)")
plt.tight_layout()
plt.show()

### 3.4 Categorical and calendar views

In [ ]:
month = df["date"].dt.month
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(x=df["date"].dt.year, ax=axes[0], color="steelblue")
axes[0].set_title("Observations per year")
sns.boxplot(x=month, y=df["pm25"], showfliers=False, ax=axes[1], color="steelblue")
axes[1].set_title("PM2.5 by calendar month (outliers hidden)")
sns.histplot(obs_per_station, bins=30, ax=axes[2])
axes[2].set_title("Observations per station")
plt.tight_layout()
plt.show()

### 3.5 Missing values

In [ ]:
print("Missing predictor values (%):")
print((df[predictors].isna().mean() * 100).round(2).to_string())

gases = ["NO2", "CO", "O3"]
print("\nNO2 / CO / O3 missing (%) by year:")
display((df[gases].isna().groupby(df["date"].dt.year).mean() * 100).round(1))

print("Stations with the most missing NO2 / CO / O3 (%):")
display((df[gases].isna().groupby(df["id"]).mean() * 100).round(1).sort_values("NO2", ascending=False).head(8))

### 3.6 Outliers

In [ ]:
rows = []
for col in analysis_cols:
    x = df[col].dropna()
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    rows.append({
        "variable": col,
        "n_out_iqr": int(((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum()),
        "n_out_z": int((np.abs(stats.zscore(x)) > 3).sum()),
        "n": len(x),
    })
outliers = pd.DataFrame(rows).set_index("variable")
outliers["pct_out_iqr"] = outliers["n_out_iqr"] / outliers["n"] * 100
outliers["pct_out_z"] = outliers["n_out_z"] / outliers["n"] * 100
outliers.round(2)

In [ ]:
# multivariate detection on standardized complete rows (exploration only, nothing is removed)
complete = df[analysis_cols].dropna()
scaled = StandardScaler().fit_transform(complete)
flag_if = IsolationForest(contamination=0.01, random_state=SEED).fit_predict(scaled) == -1
flag_lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01).fit_predict(scaled) == -1
n_out_if, n_out_lof = int(flag_if.sum()), int(flag_lof.sum())

extreme = (complete["pm25"] > 50).to_numpy()
pd.DataFrame({
    "flagged rows": [n_out_if, n_out_lof, len(complete)],
    "share with PM2.5 > 50 (%)": [extreme[flag_if].mean() * 100, extreme[flag_lof].mean() * 100, extreme.mean() * 100],
    "median PM2.5": [complete["pm25"][flag_if].median(), complete["pm25"][flag_lof].median(), complete["pm25"].median()],
}, index=["Isolation Forest", "LOF", "all rows"]).round(2)

### 3.7 Correlations and multicollinearity

In [ ]:
pearson = df[analysis_cols].corr(method="pearson")
spearman = df[analysis_cols].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(pearson, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Pearson")
sns.heatmap(spearman, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Spearman")
plt.tight_layout()
plt.show()

pairs = [(a, b, pearson.loc[a, b]) for i, a in enumerate(predictors) for b in predictors[i + 1:]
         if abs(pearson.loc[a, b]) > 0.9]
print("Predictor pairs with |r| > 0.9:", pairs if pairs else "none")
print("Largest |r| between a predictor and PM2.5:", round(pearson["pm25"].drop("pm25").abs().max(), 3))

In [ ]:
X = sm.add_constant(df[predictors].dropna())
vif = pd.Series({col: variance_inflation_factor(X.values, i)
                 for i, col in enumerate(X.columns) if col != "const"})
vif.round(2).to_frame("VIF")

### 3.8 PM2.5 and predictors: seasons, stations and meteorology

In [ ]:
season = df["date"].dt.month.map({12: "winter", 1: "winter", 2: "winter",
                                  3: "spring", 4: "spring", 5: "spring",
                                  6: "summer", 7: "summer", 8: "summer",
                                  9: "autumn", 10: "autumn", 11: "autumn"})
seasonal_corr = pd.DataFrame({
    name: g[predictors].corrwith(g["pm25"], method="spearman")
    for name, g in df.groupby(season)
})
print("Spearman correlation with PM2.5 by season:")
seasonal_corr.round(2)

In [ ]:
wind_speed = np.hypot(df["wind_u"], df["wind_v"])
meteo = {"wind speed": wind_speed, "temperature": df["temperature_celsius"], "pressure": df["pressure_mb"]}

rows = []
for name, values in meteo.items():
    level = pd.qcut(values, 3, labels=["low", "medium", "high"])
    for lv, g in df.groupby(level, observed=True):
        rows.append({"meteorology": name, "level": lv, "n": len(g),
                     "spearman AOD vs PM2.5": g["AOD"].corr(g["pm25"], method="spearman")})
print("AOD vs PM2.5 correlation inside terciles of each meteorological variable:")
pd.DataFrame(rows).round(3)

In [ ]:
station_stats = df.groupby("id")["pm25"].agg(["count", "median"])
station_stats = station_stats[station_stats["count"] >= MIN_OBS_PER_STATION]
print(f"Median PM2.5 across stations with >= {MIN_OBS_PER_STATION} observations:")
station_stats["median"].describe()[["min", "25%", "50%", "75%", "max"]].round(2)

## 4. Time-series exploration

### 4.1 Daily timeline and station diagnostics

In [ ]:
def to_daily(g):
    # regular daily timeline for one station; missing days become NaN rows
    return g.set_index("date").asfreq("D")

def adf_p(x):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return adfuller(x, autolag="AIC")[1]

def kpss_p(x):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return kpss(x, regression="c", nlags="auto")[1]

def longest_run(series):
    # longest stretch of consecutive daily observations
    ok = series.notna()
    run_id = (ok != ok.shift()).cumsum()
    return max((r for _, r in series[ok].groupby(run_id[ok])), key=len)

station_names = df.drop_duplicates("id").set_index("id")["sensor_name"]
ts_rows = []

for sid, g in df.groupby("id"):
    g = g.sort_values("date")
    pm = g["pm25"].to_numpy()
    n_days = (g["date"].max() - g["date"].min()).days + 1
    steps = g["date"].diff().dt.total_seconds().dropna()
    months = g["date"].dt.to_period("M").astype(str)
    daily_pm = to_daily(g)["pm25"]
    run = longest_run(daily_pm).to_numpy()

    # ACF on the daily timeline, using only pairs of days that were both observed
    a1, a7, a24 = [daily_pm.corr(daily_pm.shift(k), min_periods=30) for k in (1, 7, 24)]
    p1 = a1                                       # PACF at lag 1 equals ACF at lag 1

    lb = adf = kp = n_lags = np.nan
    if len(run) >= MIN_RUN_DAYS:
        pc = pacf(run, nlags=min(14, len(run) // 2 - 1), method="ywm")
        n_lags = 0
        for value in pc[1:]:                      # PACF cut-off
            if abs(value) > 1.96 / np.sqrt(len(run)):
                n_lags += 1
            else:
                break
        lb = acorr_ljungbox(run, lags=[7])["lb_pvalue"].iloc[0]
        adf, kp = adf_p(run), kpss_p(run)

    trend = stats.linregress((g["date"] - g["date"].min()).dt.days, pm) if len(pm) >= 3 else None
    month_mean = g.groupby(g["date"].dt.month)["pm25"].mean()
    half = len(pm) // 2
    drift = stats.ks_2samp(pm[:half], pm[half:]) if len(pm) >= 20 else None

    note = f"{station_names[sid]}; " + ("retained" if len(g) >= MIN_OBS_PER_STATION else f"dropped (< {MIN_OBS_PER_STATION} obs)")
    if len(run) < MIN_RUN_DAYS:
        note += f"; no ADF/KPSS/Ljung-Box (longest daily run < {MIN_RUN_DAYS} days)"

    ts_rows.append({
        "model_id": MODEL_ID, "model_group": MODEL_GROUP, "variable": "pm25_ugm3",
        "node_id": str(sid), "n_records": len(g),
        "t_min": g["date"].min().strftime("%Y-%m-%d"), "t_max": g["date"].max().strftime("%Y-%m-%d"),
        "sampling_interval_median_sec": steps.median() if len(steps) else np.nan,
        "sampling_interval_std_sec": steps.std() if len(steps) > 1 else np.nan,
        "pct_missing_intervals": (n_days - len(g)) / n_days * 100,
        "acf_lag1": a1, "acf_lag7": a7, "acf_lag24": a24, "pacf_lag1": p1,
        "ljung_box_p": lb, "adf_p": adf, "kpss_p": kp,
        "trend_slope_per_day": trend.slope if trend else np.nan,
        "trend_pval": trend.pvalue if trend else np.nan,
        "diurnal_amplitude": np.nan,
        "seasonal_amplitude": month_mean.max() - month_mean.min() if len(month_mean) >= 6 else np.nan,
        "target_drift_stat": drift.statistic if drift else np.nan,
        "target_drift_pval": drift.pvalue if drift else np.nan,
        "n_days_above_25": int((pm > 25).sum()), "n_days_above_50": int((pm > 50).sum()),
        "pct_days_above_25": (pm > 25).mean() * 100, "pct_days_above_50": (pm > 50).mean() * 100,
        "regime_breaks": np.nan,
        "cv_strategy_recommended": "temporal",
        "gap_between_train_test_sec": 86400.0,
        "n_cycles_observed": round(n_days / 365.25, 1),
        "no2_pct_missing_by_month": json.dumps(g["NO2"].isna().groupby(months).mean().mul(100).round(1).to_dict()),
        "co_pct_missing_by_month": json.dumps(g["CO"].isna().groupby(months).mean().mul(100).round(1).to_dict()),
        "o3_pct_missing_by_month": json.dumps(g["O3"].isna().groupby(months).mean().mul(100).round(1).to_dict()),
        "notes": note,
        "longest_run_days": len(run), "n_lags_recommended": n_lags,
    })

ts_table = pd.DataFrame(ts_rows)

previous_gap = df.groupby("id")["date"].diff().dt.days
print(f"Observations whose previous day is also observed: {(previous_gap == 1).mean() * 100:.1f}%")
print("Gap to previous observation (days), share of observations:")
print((previous_gap.clip(upper=7).value_counts(normalize=True).sort_index() * 100).round(1).to_string())
print()
print("Per-station summary:")
ts_table[["n_records", "pct_missing_intervals", "longest_run_days"]].describe().loc[["min", "25%", "50%", "75%", "max"]].round(1)

### 4.2 Temporal patterns and target drift

In [ ]:
monthly_mean = df.groupby(df["date"].dt.to_period("M"))["pm25"].mean()
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
monthly_mean.plot(ax=axes[0, 0], title="Monthly mean PM2.5")
df.groupby(df["date"].dt.year)["pm25"].mean().plot(marker="o", ax=axes[0, 1], title="Yearly mean PM2.5")
df.groupby(df["date"].dt.month)["pm25"].mean().plot(kind="bar", ax=axes[1, 0], title="Mean PM2.5 by calendar month")
df.groupby(df["date"].dt.dayofweek)["pm25"].mean().plot(kind="bar", ax=axes[1, 1], title="Mean PM2.5 by weekday (0 = Monday)")
for ax in axes.flat:
    ax.set_xlabel("")
    ax.set_ylabel("µg/m³")
plt.tight_layout()
plt.show()

In [ ]:
def drift_table(groups):
    return df.groupby(groups)["pm25"].agg(
        n="count",
        median="median",
        iqr=lambda x: x.quantile(0.75) - x.quantile(0.25),
        pct_above_25=lambda x: (x > 25).mean() * 100,
        pct_above_50=lambda x: (x > 50).mean() * 100,
    ).round(2)

print("By year:")
display(drift_table(df["date"].dt.year.rename("year")))
print("By calendar month:")
drift_table(df["date"].dt.month.rename("month"))

In [ ]:
monthly_predictors = df.groupby(df["date"].dt.month)[predictors + ["pm25"]].mean()
standardized = (monthly_predictors - monthly_predictors.mean()) / monthly_predictors.std()
standardized.plot(figsize=(11, 4), marker="o")
plt.title("Seasonal cycle of PM2.5 and predictors (standardized monthly means)")
plt.xlabel("month")
plt.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

### 4.3 Autocorrelation and stationarity

In [ ]:
best = ts_table.loc[ts_table["longest_run_days"].idxmax()]
g_best = df[df["id"] == int(best["node_id"])].sort_values("date")
run = longest_run(to_daily(g_best)["pm25"]).to_numpy()
max_lag = min(30, len(run) // 2 - 1)
print(f"Station {station_names[int(best['node_id'])]}: longest daily run = {len(run)} days")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(run, lags=max_lag, ax=axes[0])
plot_pacf(run, lags=max_lag, method="ywm", ax=axes[1])
plt.tight_layout()
plt.show()

print("ADF p-value:", round(adf_p(run), 4))
print("Ljung-Box p-value (lag 7):", round(acorr_ljungbox(run, lags=[7])["lb_pvalue"].iloc[0], 4))

In [ ]:
with_acf = ts_table.dropna(subset=["acf_lag1"])
print(f"ACF (pairs of observed days): {len(with_acf)} of {len(ts_table)} stations")
print(with_acf[["acf_lag1", "acf_lag7", "acf_lag24"]].describe().loc[["25%", "50%", "75%"]].round(2))

tested = ts_table.dropna(subset=["ljung_box_p"])
print(f"\nTests on the longest daily run: {len(tested)} stations with a run of at least {MIN_RUN_DAYS} days")
print(f"Ljung-Box p < 0.05: {(tested['ljung_box_p'] < 0.05).mean() * 100:.0f}% of these stations")
print(f"ADF p < 0.05 (stationary): {(tested['adf_p'] < 0.05).mean() * 100:.0f}%")
print(f"KPSS p > 0.05 (stationary): {(tested['kpss_p'] > 0.05).mean() * 100:.0f}%")
print(f"PACF cut-off (median number of lags): {tested['n_lags_recommended'].median():.0f}")

### 4.4 Lead–lag relationships

In [ ]:
lag_vars = ["pm25", "AOD", "NO2", "CO", "O3", "temperature_celsius", "wind_u", "wind_v"]
daily = pd.concat([to_daily(g.sort_values("date"))[lag_vars].assign(id=sid)
                   for sid, g in df.groupby("id")]).reset_index(drop=True)

lead_lag = pd.DataFrame({
    var: [daily["pm25"].corr(daily.groupby("id")[var].shift(k), method="spearman") for k in range(8)]
    for var in lag_vars
}, index=[f"t-{k}" for k in range(8)]).T
lead_lag.loc["pm25", "t-0"] = np.nan

plt.figure(figsize=(8, 4))
sns.heatmap(lead_lag, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Spearman correlation between PM2.5 at day t and a variable at day t-k")
plt.tight_layout()
plt.show()

### 4.5 Decision on temporal validation and lag features

In [ ]:
observed = daily["pm25"].notna()
print("Share of observations whose previous days are also observed:")
for w in (2, 3, 7):
    full = observed.groupby(daily["id"]).transform(lambda s: s.rolling(w).sum() == w)
    print(f"  previous {w - 1} day(s): {full[observed].mean() * 100:.1f}%")
print(f"\nMedian ACF at lag 1 over stations: {ts_table['acf_lag1'].median():.2f}")

## 5. Preprocessing

### 5.1 Station selection
Stations with fewer than 100 valid days cannot support temporal or spatial analysis, so they are dropped before the split (15 stations, 590 rows).

In [ ]:
counts = df.groupby("id").size()
keep_ids = counts[counts >= MIN_OBS_PER_STATION].index
df_model = df[df["id"].isin(keep_ids)]

print(f"Stations kept: {len(keep_ids)} of {len(counts)} (at least {MIN_OBS_PER_STATION} valid days)")
print(f"Rows kept: {len(df_model):,} of {len(df):,}")
print("Dropped stations:", ", ".join(sorted(station_names[counts[counts < MIN_OBS_PER_STATION].index])))

### 5.2 Feature construction

In [ ]:
parts = []
for sid, g in df_model.groupby("id"):
    g = to_daily(g.sort_values("date"))      # lags are calendar-day lags: NaN across gaps
    g["id"] = sid
    g["pm25_lag1"] = g["pm25"].shift(1)
    for k in (1, 2, 3):
        g[f"AOD_lag{k}"] = g["AOD"].shift(k)
    g["pm25_roll7_mean"] = g["pm25"].shift(1).rolling(7, min_periods=3).mean()   # past values only
    parts.append(g)

data = pd.concat(parts).dropna(subset=["pm25"]).reset_index()
day_of_year = data["date"].dt.dayofyear
data["doy_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
data["doy_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)
for col in ["NO2", "CO", "O3"]:
    data[f"{col}_missing"] = data[col].isna().astype(int)

FEATURES = (predictors
            + ["doy_sin", "doy_cos"]
            + ["pm25_lag1", "AOD_lag1", "AOD_lag2", "AOD_lag3", "pm25_roll7_mean"]
            + ["NO2_missing", "CO_missing", "O3_missing"])
TARGET = "pm25"

print("Number of features:", len(FEATURES))
print("Share of missing values per feature (%):")
(data[FEATURES].isna().mean() * 100).round(1)[lambda s: s > 0].to_frame("missing %")

### 5.3 Development / test split

In [ ]:
dev = data[data["date"] < TEST_START]
test = data[data["date"] >= TEST_START]

assert dev["date"].max() < test["date"].min()

pd.DataFrame({
    "rows": [len(dev), len(test)],
    "stations": [dev["id"].nunique(), test["id"].nunique()],
    "first_date": [dev["date"].min().date(), test["date"].min().date()],
    "last_date": [dev["date"].max().date(), test["date"].max().date()],
    "mean_pm25": [dev["pm25"].mean(), test["pm25"].mean()],
    "pct_above_25": [(dev["pm25"] > 25).mean() * 100, (test["pm25"] > 25).mean() * 100],
}, index=["train", "test"]).round(2)

In [ ]:
new_stations = set(test["id"]) - set(dev["id"])
print("Stations that appear only in the test period:", len(new_stations))

### 5.4 Preprocessing pipeline

In [ ]:
preprocessing = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessing.fit(dev[FEATURES])
X_test = preprocessing.transform(test[FEATURES])
print("Test matrix:", X_test.shape, "| NaN left:", int(np.isnan(X_test).sum()))

## 6. Output files

In [ ]:
EXPLORATORY_DATA_COLUMNS = [
    "model_id", "model_group", "variable", "family", "type", "unit", "n_total", "n_missing",
    "pct_missing", "n_unique", "n_duplicates_key", "min", "q1", "median", "q3", "max", "mean",
    "std", "skewness", "kurtosis", "iqr", "n_out_iqr", "pct_out_iqr", "n_out_z", "pct_out_z",
    "n_out_if", "pct_out_if", "n_out_lof", "pct_out_lof", "n_sentinel_low", "n_sentinel_high",
    "n_removed_total", "max_abs_corr", "corr_partner", "vif", "modalities", "target_assoc",
    "target_pval", "acf_lag1", "acf_lag24", "pacf_lag1", "n_lags_recommended", "adf_p",
    "is_leakage_suspect", "notes",
]
EXPLORATORY_TEMPORAL_COLUMNS = [
    "model_id", "model_group", "variable", "node_id", "n_records", "t_min", "t_max",
    "sampling_interval_median_sec", "sampling_interval_std_sec", "pct_missing_intervals",
    "acf_lag1", "acf_lag7", "acf_lag24", "pacf_lag1", "ljung_box_p", "adf_p", "kpss_p",
    "trend_slope_per_day", "trend_pval", "diurnal_amplitude", "seasonal_amplitude",
    "target_drift_stat", "target_drift_pval", "n_days_above_25", "n_days_above_50",
    "pct_days_above_25", "pct_days_above_50", "regime_breaks", "cv_strategy_recommended",
    "gap_between_train_test_sec", "n_cycles_observed", "no2_pct_missing_by_month",
    "co_pct_missing_by_month", "o3_pct_missing_by_month", "notes",
]

families = {"pm25": "Target", "AOD": "AOD", "NO2": "NO2", "CO": "CO", "O3": "O3",
            "temperature_celsius": "Meteorological", "pressure_mb": "Meteorological",
            "wind_u": "Meteorological", "wind_v": "Meteorological",
            "latitude": "Spatial", "longitude": "Spatial",
            "date": "Calendar", "id": "ID", "sensor_name": "ID"}
notes = {
    "pm25": "target; extreme values kept; IF/LOF = multivariate on 9 variables (1%); ACF/PACF = median over stations, ADF/lags = median over stations with a daily run >= 20 days",
    "NO2": "missing mostly before 2019 (Sentinel-5P period)",
    "CO": "missing mostly before 2019 (Sentinel-5P period)",
    "O3": "missing mostly before 2019 (Sentinel-5P period)",
    "latitude": "not a predictive feature",
    "longitude": "not a predictive feature",
    "id": "station key (GEE id); not a predictive feature",
    "sensor_name": "name only; 4 names have 2 ids, so id is the station key; not a predictive feature",
}
n_duplicates_key = int(df.duplicated(["date", "latitude", "longitude", "id"]).sum())

rows = []
for col in ["pm25"] + predictors + ["latitude", "longitude", "date", "id", "sensor_name"]:
    s = df[col]
    kind = "id" if col in ("id", "sensor_name") else ("datetime" if col == "date" else "numeric")
    row = {
        "model_id": MODEL_ID, "model_group": MODEL_GROUP,
        "variable": "pm25_ugm3" if col == "pm25" else col,
        "family": families[col], "type": kind, "unit": units.get(col, ""),
        "n_total": len(df), "n_missing": int(s.isna().sum()), "pct_missing": s.isna().mean() * 100,
        "n_unique": int(s.nunique()), "n_duplicates_key": n_duplicates_key,
        "n_removed_total": n_removed, "is_leakage_suspect": False, "notes": notes.get(col, ""),
    }
    if kind == "id":
        row["modalities"] = int(s.nunique())
    if kind == "numeric":
        x = s.dropna()
        q1, med, q3 = x.quantile([0.25, 0.5, 0.75])
        row.update(min=x.min(), q1=q1, median=med, q3=q3, max=x.max(), mean=x.mean(), std=x.std(),
                   skewness=x.skew(), kurtosis=x.kurtosis(), iqr=q3 - q1)
        if col in outliers.index:
            row.update(n_out_iqr=outliers.loc[col, "n_out_iqr"], pct_out_iqr=outliers.loc[col, "pct_out_iqr"],
                       n_out_z=outliers.loc[col, "n_out_z"], pct_out_z=outliers.loc[col, "pct_out_z"])
        if col in pearson.columns:
            partner = pearson[col].drop(col).abs().idxmax()
            row["corr_partner"], row["max_abs_corr"] = partner, abs(pearson.loc[partner, col])
        if col in vif.index:
            row["vif"] = vif[col]
        if col != "pm25":
            pair = df[[col, "pm25"]].dropna()
            r, p = stats.pearsonr(pair[col], pair["pm25"])
            rho, _ = stats.spearmanr(pair[col], pair["pm25"])
            row["target_assoc"] = f"pearson_r={r:.3f}; spearman={rho:.3f}"
            row["target_pval"] = p
            row["is_leakage_suspect"] = bool(abs(r) > 0.95)
    if col == "pm25":
        row.update(n_sentinel_low=n_low, n_sentinel_high=n_high,
                   n_out_if=n_out_if, pct_out_if=n_out_if / len(complete) * 100,
                   n_out_lof=n_out_lof, pct_out_lof=n_out_lof / len(complete) * 100,
                   acf_lag1=ts_table["acf_lag1"].median(), acf_lag24=ts_table["acf_lag24"].median(),
                   pacf_lag1=ts_table["pacf_lag1"].median(), adf_p=tested["adf_p"].median(),
                   n_lags_recommended=int(tested["n_lags_recommended"].median()))
    rows.append(row)

exploratory_data = pd.DataFrame(rows).reindex(columns=EXPLORATORY_DATA_COLUMNS)
count_cols = [c for c in exploratory_data.columns if c.startswith("n_")]
exploratory_data[count_cols] = exploratory_data[count_cols].astype("Int64")
exploratory_temporal = ts_table.reindex(columns=EXPLORATORY_TEMPORAL_COLUMNS)

exploratory_data.to_csv(OUTPUT_DIR / "Exploratory_Data.csv", index=False)
exploratory_temporal.to_csv(OUTPUT_DIR / "Exploratory_temporal.csv", index=False)

print("Exploratory_Data.csv:    ", exploratory_data.shape)
print("Exploratory_temporal.csv:", exploratory_temporal.shape)
exploratory_data.head(3)